In [0]:
from pyspark.sql import functions as F

base = "/Volumes/workspace/raw/synthea_csv"

# Files we're bringing into bronze
source_files = [
    "patients",
    "encounters",
    "conditions",
    "medications",
    "procedures",
    "organizations",
]

for name in source_files:
    file_path = f"{base}/{name}.csv"

    df = (
        spark.read
        .option("header", True)
        .option("inferSchema", True)   # fine for bronze; we'll enforce types in silver
        .csv(file_path)
        .withColumn("ingest_timestamp", F.current_timestamp())
        .withColumn("source_file", F.lit(f"{name}.csv"))
    )

    target_table = f"workspace.bronze.{name}"

    (
        df.write
        .mode("overwrite")          # rerunning this notebook re-loads cleanly
        .option("overwriteSchema", "true")
        .saveAsTable(target_table)
    )

    print(f"{target_table}: {df.count()} rows written")

In [0]:
%sql
SHOW TABLES IN workspace.bronze

In [0]:
for name in source_files:
    cnt = spark.table(f"workspace.bronze.{name}").count()
    print(f"{name}: {cnt} rows")

In [0]:
spark.table("workspace.bronze.encounters").groupBy("ENCOUNTERCLASS").count().show()